<a href="https://colab.research.google.com/github/pvenkatesh282675-ops/AI-Resume-Screening-System/blob/main/Nm_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from IPython import get_ipython
from IPython.display import display
!pip install streamlit yfinance pandas numpy matplotlib tensorflow scikit-learn statsmodels prophet
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet
import datetime
import streamlit as st # Import streamlit and alias it as st

# App title and description
st.title('📈 Stock Market Forecasting App')
st.write("""This app predicts future stock prices using different forecasting models.
Search for a company by ticker symbol and select a prediction date.
""")

# Sidebar for user input
st.sidebar.header('User Input Parameters')

# Function to get company info
def get_company_info(ticker):
    company = yf.Ticker(ticker)
    info = company.info
    return info

# Search for company by ticker
def search_company():
    ticker = st.sidebar.text_input('Enter company ticker (e.g., AAPL):', 'AAPL')
    try:
        info = get_company_info(ticker)
        st.sidebar.success(f"Found: {info.get('longName', 'N/A')}")
        st.sidebar.write(f"Sector: {info.get('sector', 'N/A')}")
        st.sidebar.write(f"Industry: {info.get('industry', 'N/A')}")
        return ticker
    except:
        st.sidebar.error("Company not found. Please try another ticker.")
        return None

# Date selection for prediction
def select_prediction_date():
    today = datetime.date.today()
    future_date = st.sidebar.date_input(
        "Select prediction date:",
        today + datetime.timedelta(days=30),
        min_value=today + datetime.timedelta(days=1),
        max_value=today + datetime.timedelta(days=365*2))  # 2 years max
    return future_date

# Download stock data
@st.cache_data
def load_data(ticker, start_date='2010-01-01'):
    data = yf.download(ticker, start=start_date)
    return data

# Preprocess data for LSTM
def preprocess_data(data, n_steps=60):
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(data['Close'].values.reshape(-1, 1))

    X, y = [], []
    for i in range(n_steps, len(scaled_data)):
        X.append(scaled_data[i-n_steps:i, 0])
        y.append(scaled_data[i, 0])

    X, y = np.array(X), np.array(y)
    X = np.reshape(X, (X.shape[0], X.shape[1], 1))
    return X, y, scaler

# Build LSTM model
def build_lstm_model(input_shape):
    model = Sequential()
    model.add(LSTM(units=50, return_sequences=True, input_shape=input_shape))
    model.add(LSTM(units=50))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

# ARIMA model
def arima_forecast(data, steps):
    model = ARIMA(data['Close'], order=(5,1,0))
    model_fit = model.fit()
    forecast = model_fit.forecast(steps=steps)
    return forecast

# Prophet model
def prophet_forecast(data, periods):
    df = data.reset_index()[['Date', 'Close']].rename(columns={'Date': 'ds', 'Close': 'y'})
    model = Prophet(daily_seasonality=True)
    model.fit(df)
    future = model.make_future_dataframe(periods=periods)
    forecast = model.predict(future)
    return forecast

# Main app function
def main():
    # Get user inputs
    ticker = search_company()
    if not ticker:
        return

    prediction_date = select_prediction_date()
    today = datetime.date.today()
    days_to_predict = (prediction_date - today).days

    # Load data
    data = load_data(ticker)
    if data.empty:
        st.error("No data available for this ticker. Please try another one.")
        return

    # Show raw data
    st.subheader(f'Raw Data for {ticker}')
    st.write(data.tail())

    # Plot closing price
    st.subheader('Closing Price History')
    fig, ax = plt.subplots(figsize=(16,8))
    ax.plot(data['Close'], label='Close Price')
    ax.set_title('Stock Price History')
    ax.set_xlabel('Date')
    ax.set_ylabel('Price (USD)')
    st.pyplot(fig)

    # Model selection
    st.subheader('Select Forecasting Models')
    col1, col2, col3 = st.columns(3)
    use_lstm = col1.checkbox('LSTM', value=True)
    use_arima = col2.checkbox('ARIMA', value=True)
    use_prophet = col3.checkbox('Prophet', value=True)

    if not (use_lstm or use_arima or use_prophet):
        st.warning("Please select at least one forecasting model")
        return

    # Forecasting
    st.subheader('Forecasting Results')

    # Calculate predictions for each selected model
    predictions = {}

    # LSTM Model
    if use_lstm:
        with st.spinner('Training LSTM model...'):
            try:
                # Prepare data
                n_steps = 60
                X, y, scaler = preprocess_data(data, n_steps)

                # Split data
                train_size = int(len(X) * 0.8)
                X_train, X_test = X[:train_size], X[train_size:]
                y_train, y_test = y[:train_size], y[train_size:]

                # Build and train model
                model = build_lstm_model((X_train.shape[1], 1))
                model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)

                # Make prediction
                last_sequence = X_test[-1]
                predictions_lstm = []
                for _ in range(days_to_predict):
                    next_pred = model.predict(last_sequence.reshape(1, n_steps, 1))
                    predictions_lstm.append(next_pred[0,0])
                    last_sequence = np.append(last_sequence[1:], next_pred[0,0]).reshape(n_steps, 1)

                # Inverse transform
                predictions_lstm = scaler.inverse_transform(np.array(predictions_lstm).reshape(-1, 1))
                predictions['LSTM'] = predictions_lstm.flatten()
                st.success("LSTM model trained successfully!")
            except Exception as e:
                st.error(f"LSTM model failed: {str(e)}")

    # ARIMA Model
    if use_arima:
        with st.spinner('Training ARIMA model...'):
            try:
                forecast_arima = arima_forecast(data, days_to_predict)
                predictions['ARIMA'] = forecast_arima.values
                st.success("ARIMA model trained successfully!")
            except Exception as e:
                st.error(f"ARIMA model failed: {str(e)}")

    # Prophet Model
    if use_prophet:
        with st.spinner('Training Prophet model...'):
            try:
                forecast_prophet = prophet_forecast(data, days_to_predict)
                predictions['Prophet'] = forecast_prophet.tail(days_to_predict)['yhat'].values
                st.success("Prophet model trained successfully!")
            except Exception as e:
                st.error(f"Prophet model failed: {str(e)}")

    # Display predictions
    if predictions:
        # Create date range for predictions
        last_date = data.index[-1]
        prediction_dates = pd.date_range(start=last_date + datetime.timedelta(days=1), periods=days_to_predict)

        # Plot predictions
        fig, ax = plt.subplots(figsize=(16,8))
        ax.plot(data.index, data['Close'], label='Historical Data')

        for model_name, preds in predictions.items():
            ax.plot(prediction_dates, preds, label=f'{model_name} Prediction')

        ax.set_title(f'Stock Price Prediction until {prediction_date}')
        ax.set_xlabel('Date')
        ax.set_ylabel('Price (USD)')
        ax.legend()
        st.pyplot(fig)

        # Show prediction for selected date
        st.subheader(f'Predicted Price on {prediction_date}')

        result_df = pd.DataFrame({
            'Date': prediction_dates,
            **{f'{model}_Prediction': preds for model, preds in predictions.items()}
        })

        # Find the row for the selected prediction date
        selected_prediction = result_df[result_df['Date'].dt.date == prediction_date]

        if not selected_prediction.empty:
            st.dataframe(selected_prediction.set_index('Date').T)
        else:
            st.warning("No prediction available for the selected date")
    else:
        st.warning("No models produced successful predictions")

if __name__ == '__main__':
    main()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 28.6 MB/s eta 0:00:00
